# Gaia BP/RP Spectra Downloader (Generic)

This notebook:
1. Reads `source_id` from any FITS file (`sample_stars.fits`)
2. Checks which sources have BP/RP sampled spectra in Gaia DR3
3. Downloads them in batches to `./BPRP_spectra/<source_id>.fits`

## Setup and Imports

In [13]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astroquery.gaia import Gaia
from pathlib import Path
import os

# Directory for downloaded spectra
spectra_dir = Path('./BPRP_spectra')
spectra_dir.mkdir(exist_ok=True)

print(f"Spectra will be saved to: {spectra_dir.absolute()}")

Spectra will be saved to: /Users/rix/Science/Projects/GAIA/GaiaDR3/BP-RP/All-Sky/HotStars_XP/BPRP_spectra


## Optional: Login to Gaia Archive (recommended for faster/reliable access)

In [14]:
# Login to Gaia archive
username =  '<name>' #input("Enter your Gaia archive username: ")
password = '<passwd>'  #getpass.getpass("Enter your Gaia archive password: ")

try:
    Gaia.login(user=username, password=password)
    print("✓ Successfully logged in to Gaia archive")
except Exception as e:
    print(f"Login failed: {e}")
    print("Note: You can continue without login for public data access")

INFO: Login to gaia TAP server [astroquery.gaia.core]
401 Error 401:
<!doctype html><html lang="en"><head><title>HTTP Status 401 – Unauthorized</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 401 – Unauthorized</h1><hr class="line" /><p><b>Type</b> Status Report</p><p><b>Message</b> Bad Credentials</p><p><b>Description</b> The request has not been applied to the target resource because it lacks valid authentication credentials for that resource.</p><hr class="line" /><h3>Apache Tomcat/9.0.102</h3></body></html>
✓ Successfully logged in to Gaia archive


ERROR: Error logging in TAP server [astroquery.gaia.core]


## Load source_id from FITS file

In [16]:
fits_file = 'sample_stars.fits'
fits_file = 'Zari.G.lt.12.fits'
fits_file ='JMH.SDSSV_massive_subsample.fits'


with fits.open(fits_file) as hdul:
    data = hdul[1].data
    source_ids = np.unique(data['source_id'].astype(np.int64))  # ensure unique & int64

print(f"Loaded {len(source_ids)} unique source_ids from {fits_file}")
print(f"First 10: {source_ids[:10]}")

Loaded 1679 unique source_ids from JMH.SDSSV_massive_subsample.fits
First 10: [246266350946251904 246282774900821376 251695773724432896
 251813833785517824 251814555340747008 252940107349231872
 254040482269101952 254502135416485888 254502238495700736
 257432922319662592]


## Check which sources have BP/RP sampled spectra

In [17]:
# FIXED & WORKING cell – replace your current check_bprp_availability cell with this one
def check_bprp_availability(source_ids, batch_size=2000):
    sources_with_spectra = []
    print("Checking which sources have BP/RP sampled spectra...")
    
    for i in range(0, len(source_ids), batch_size):
        batch = source_ids[i:i+batch_size]
        id_list = ','.join(map(str, batch))
        
        # THIS IS THE CORRECT WAY for the public archive (anonymous or logged-in)
        query = f"""
        SELECT source_id
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({id_list})
          AND has_xp_sampled = 'true'
        """
        
        try:
            job = Gaia.launch_job(query)
            result = job.get_results()
            sources_with_spectra.extend(result['source_id'].data)
            print(f"  Batch {i//batch_size + 1}/{(len(source_ids)-1)//batch_size + 1}: "
                  f"{len(result)} sources have sampled spectra")
        except Exception as e:
            print(f"  Batch failed (will skip): {e}")
    
    return np.array(sources_with_spectra, dtype=np.int64)


# Run it
sources_with_spectra = check_bprp_availability(source_ids)

print(f"\nDone → {len(sources_with_spectra)} / {len(source_ids)} "
      f"sources have BP/RP sampled spectra ({100*len(sources_with_spectra)/len(source_ids):.1f}%)")

Checking which sources have BP/RP sampled spectra...
  Batch 1/1: 1559 sources have sampled spectra

Done → 1559 / 1679 sources have BP/RP sampled spectra (92.9%)


## Download BP/RP spectra in batches of ~100

In [18]:
batch_size = 50
n_batches = (len(sources_with_spectra) + batch_size - 1) // batch_size

print(f"\nDownloading {len(sources_with_spectra)} spectra in {n_batches} batches of ~{batch_size}...\n")

for batch_idx in range(0, len(sources_with_spectra), batch_size):
    batch_ids = sources_with_spectra[batch_idx:batch_idx + batch_size]
    print(f"Batch {(batch_idx//batch_size)+1}/{n_batches} – downloading {len(batch_ids)} sources...")
    
    try:
        datalink = Gaia.load_data(ids=batch_ids, retrieval_type='XP_SAMPLED', data_structure='INDIVIDUAL', verbose=False)
        
        for key, prod_list in datalink.items():
            # Extract source_id from filename/key
            source_id = None
            for sid in batch_ids:
                if str(sid) in key:
                    source_id = sid
                    break
            if source_id is None:
                continue
                
            # Handle list or single product
            prod = prod_list[0] if isinstance(prod_list, list) else prod_list
            table = prod.to_table() if hasattr(prod, 'to_table') else prod
            
            out_file = spectra_dir / f"{source_id}.fits"
            table.write(out_file, overwrite=True)
            #print(f"  → Saved {source_id}.fits")
            
    except Exception as e:
        print(f"  [Warning] Batch failed: {e}. Falling back to individual downloads...")
        # Individual fallback
        for sid in batch_ids:
            try:
                dl = Gaia.load_data(ids=[sid], retrieval_type='XP_SAMPLED', data_structure='INDIVIDUAL', verbose=False)
                for k, pl in dl.items():
                    if str(sid) not in k:
                        continue
                    prod = pl[0] if isinstance(pl, list) else pl
                    tab = prod.to_table() if hasattr(prod, 'to_table') else prod
                    out_file = spectra_dir / f"{sid}.fits"
                    tab.write(out_file, overwrite=True)
                    #print(f"    → Saved {sid}.fits (individual)")
                    break
            except Exception as e2:
                print(f"    [Error] Failed {sid}: {e2}")

print(f"\nAll done! Spectra saved in: {spectra_dir}")



Batch 1/32 – downloading 50 sources...
Batch 2/32 – downloading 50 sources...
Batch 3/32 – downloading 50 sources...
Batch 4/32 – downloading 50 sources...
Batch 5/32 – downloading 50 sources...
Batch 6/32 – downloading 50 sources...
Batch 7/32 – downloading 50 sources...
Batch 8/32 – downloading 50 sources...
Batch 9/32 – downloading 50 sources...
Batch 10/32 – downloading 50 sources...
Batch 11/32 – downloading 50 sources...
Batch 12/32 – downloading 50 sources...
Batch 13/32 – downloading 50 sources...
Batch 14/32 – downloading 50 sources...
Batch 15/32 – downloading 50 sources...
Batch 16/32 – downloading 50 sources...
Batch 17/32 – downloading 50 sources...
Batch 18/32 – downloading 50 sources...
Batch 19/32 – downloading 50 sources...
Batch 20/32 – downloading 50 sources...
Batch 21/32 – downloading 50 sources...
Batch 22/32 – downloading 50 sources...
Batch 23/32 – downloading 50 sources...
Batch 24/32 – downloading 50 sources...
Batch 25/32 – downloading 50 sources...
Batch 2

## Optional: Logout

In [ ]:
# Gaia.logout()